

Dataset - https://github.com/YBIFoundation/Dataset/raw/main/Cancer.csv 

### librerias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')




### Analisis y carga del dataset

In [ ]:
df = pd.read_csv ('./data/cancer.csv')

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.columns

### Entrenamiento y puntuaci?n de los modelos ML

In [ ]:
y = df['diagnosis']
X = df.drop(['id','diagnosis','Unnamed: 32'],axis=1)

In [ ]:
def model_test_classification(x, y, use_cross_val=False, cv=5, plot=True, average='binary', pos_label='M'):

    if y.dtype == 'O' or isinstance(y.iloc[0], str) or isinstance(y[0], str):
        le = LabelEncoder()
        y_encoded = le.fit_transform(y)
        pos_label_num = le.transform([pos_label])[0]
    else:
        y_encoded = y
        pos_label_num = pos_label

    models = [
        LogisticRegression(max_iter=5000),
        DecisionTreeClassifier(),
        RandomForestClassifier(),
        GradientBoostingClassifier(),
        GaussianNB(),
        SVC(probability=True),
        KNeighborsClassifier(),
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        LGBMClassifier(),
        CatBoostClassifier(verbose=0)
    ]

    model_names = [
        "Logistic Regression", "Decision Tree", "Random Forest", "Gradient Boosting",
        "Naive Bayes", "SVM", "KNN", "XGBoost", "LightGBM", "CatBoost"
    ]

    # Definimos una paleta con un color distinto por modelo
    colores = sns.color_palette("tab10", n_colors=len(models))

    acc_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    roc_auc_scores = []

    x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

    for model in models:
        if use_cross_val:
            y_pred = cross_val_predict(model, x, y_encoded, cv=cv, method='predict')
            if hasattr(model, "predict_proba"):
                y_proba = cross_val_predict(model, x, y_encoded, cv=cv, method='predict_proba')[:, 1]
            else:
                y_proba = None
        else:
            model.fit(x_train, y_train)
            y_pred = model.predict(x_test)
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(x_test)[:, 1]
            else:
                y_proba = None

        y_true = y_test if not use_cross_val else y_encoded

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average=average, zero_division=0, pos_label=pos_label_num)
        rec = recall_score(y_true, y_pred, average=average, zero_division=0, pos_label=pos_label_num)
        f1 = f1_score(y_true, y_pred, average=average, zero_division=0, pos_label=pos_label_num)

        if y_proba is not None and len(np.unique(y_true)) == 2:
            roc_auc = roc_auc_score(y_true, y_proba)
        else:
            roc_auc = np.nan

        acc_scores.append(np.round(acc * 100, 2))
        precision_scores.append(np.round(prec * 100, 2))
        recall_scores.append(np.round(rec * 100, 2))
        f1_scores.append(np.round(f1 * 100, 2))
        roc_auc_scores.append(np.round(roc_auc * 100, 2) if not np.isnan(roc_auc) else None)

    results = pd.DataFrame({
        "Model": model_names,
        "Accuracy (%)": acc_scores,
        "Precision (%)": precision_scores,
        "Recall (%)": recall_scores,
        "F1 Score (%)": f1_scores,
        "ROC AUC (%)": roc_auc_scores
    }).sort_values(by="F1 Score (%)", ascending=False)

    print(results)

    if plot:
        metrics_to_plot = ["Accuracy (%)", "Precision (%)", "Recall (%)", "F1 Score (%)"]
        if results["ROC AUC (%)"].notna().all():
            metrics_to_plot.append("ROC AUC (%)")

        for metric in metrics_to_plot:
            plt.figure(figsize=(10, 5))
            # Usamos la paleta definida, ordenada seg?n los modelos en el DataFrame:
            sns.barplot(data=results, x="Model", y=metric, palette=colores)
            plt.title(f"Model Comparison - {metric}")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()

    return results


In [ ]:
model_test_classification(X, y, use_cross_val=False, plot=True)


### Modelo final 

In [ ]:
# Codificar etiquetas de y si son texto
le = LabelEncoder()
y_encoded = le.fit_transform(y)  # convierte 'B','M' a 0,1

# Dividir en entrenamiento y prueba
x_train, x_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Crear el modelo XGBoost
model_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Entrenar
model_xgb.fit(x_train, y_train)

# Predecir
y_pred = model_xgb.predict(x_test)

# Evaluar
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Matriz de confusi?n
cm = confusion_matrix(y_test, y_pred)

# Visualizaci?n
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - XGBoost')
plt.tight_layout()
plt.show()
